<a href="https://colab.research.google.com/github/Plumz17/PP_Assignment01/blob/main/PP_Assignment01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#First Pattern Recognition Assignment - Text Preprocessing and Representation - Anders Emmanuel Tan (24/541351/PA/22964)

Description: In this Assignment, I will be implementing several Text Preprocessing methods and then representating the resulting tokens into TF-IDF numeric representation.

(Note: No LLMs are used during this assignment, except for generating the stop word list)


## 0. Preparing the Text
Since the uploaded files in Google Colab are only available in the runtime, I will be attaining those files by cloning them from the Github Submission link. The Following code will be used to obtain the data named "dataset.txt" from my github repository and clone them into the "PP_Assignment01" folder in google colab.

In [2]:
# Cloning Github Repository to get the Text
!rm -rf PP_Assignment01
!git clone https://github.com/Plumz17/PP_Assignment01

Cloning into 'PP_Assignment01'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), done.
Resolving deltas: 100% (1/1), done.
['The stars are gone.', 'The planets have disappeared.', 'They were barely enough of us left to give "the end" a name.', 'The Quiet Rapture.', 'Supplies dwindle.', 'Infrastructure crumbles.', 'Too few to rebuild.', 'Too many to feed.', 'Humanity decays.', "But don't despair, my sons.", 'I tell you now that there is more to these moons than meets the eye.', 'The Consolidation, They hide their technology.', "Their people, They won't tell you what they found, But I will.", 'One moon stands apart from the rest.', 'And in the darkness of that moon, An ocean of blood.']


## 1. Text Preprocessing
Description: In the first section, we will be implementing several text preprocessing methods which will transform the input documents include several index tokens representing the documents itself. We will use the following methods:


*   Parsing
*   Lexical Analysis (Case Folding)
*   Lexical Analysis (Cleaning)
*   Lexical Analysis (Tokenization)
*   Stop-Word Removal
*   Phrase Detection
*   Stemming
*   Lemmatization

Phrase Detection, Stemming, and Lemmatization will not be included since those methods need external libraries to work reliably.


##1A. Parsing
Description: Parsing is part of the Text Preprocessing pipeline where the document structure is converted into seperated components. In my implementation, each sentence is treated as one document unit. We can achieve this by creating an empty list and then reading the previously imported dataset, line by line using a for loop. For each line, we will call the .strip() function to remove leading and ending whitespace characters, including newline characters (\n). If the resulting line is not empty, it is appended to the list as a separate document. The resulting variable, refered here as "documents" is an array containing sentences from the dataset.

In [18]:
def parse_documents(file_path):
  documents = []
  with open(file_path, "r") as file:
      for line in file:
        stripped_line = line.strip()
        if stripped_line != "":
          documents.append(stripped_line)
  return documents

documents = parse_documents("PP_Assignment01/dataset.txt")
print(documents)

['The stars are gone.', 'The planets have disappeared.', 'They were barely enough of us left to give "the end" a name.', 'The Quiet Rapture.', 'Supplies dwindle.', 'Infrastructure crumbles.', 'Too few to rebuild.', 'Too many to feed.', 'Humanity decays.', "But don't despair, my sons.", 'I tell you now that there is more to these moons than meets the eye.', 'The Consolidation, They hide their technology.', "Their people, They won't tell you what they found, But I will.", 'One moon stands apart from the rest.', 'And in the darkness of that moon, An ocean of blood.']


##1B. Lexical Analysis
Description: Lexical Analysis is a stage in the text preprocessing pipeline where the raw text from our dataset is then transformed into smaller meaningful units called token. In this implementation lexicial analysis consists of three main steps:


1.   Case Folding: For Consistency, all characters are converted to lower case.
2.   Cleaning: Non-Alphabetic characters such as punctuation marks, numbers, and other special symbols are removed.
3.   Tokenization: The Cleaned texts are split into individual words using their whitespace as the delimiter, each resulting words is called a "token"

The output of the function below is the tokenized document of the input. We can use a for loop to apply lexical analysis to our documents that we will store in "tokenized_documents"

In [21]:
def lexical_analysis(document):
  # 1. Case Folding
  document = document.lower()

  # 2. Cleaning
  cleaned_text = ""
  for char in document:
    if char.isalpha() or char.isspace():
      cleaned_text += char

  # 3. Tokenization
  tokens = cleaned_text.split()
  return tokens

# Apply Function
tokenized_documents = []
for doc in documents:
  tokens = lexical_analysis(doc)
  tokenized_documents.append(tokens)

print(tokenized_documents)

[['the', 'stars', 'are', 'gone'], ['the', 'planets', 'have', 'disappeared'], ['they', 'were', 'barely', 'enough', 'of', 'us', 'left', 'to', 'give', 'the', 'end', 'a', 'name'], ['the', 'quiet', 'rapture'], ['supplies', 'dwindle'], ['infrastructure', 'crumbles'], ['too', 'few', 'to', 'rebuild'], ['too', 'many', 'to', 'feed'], ['humanity', 'decays'], ['but', 'dont', 'despair', 'my', 'sons'], ['i', 'tell', 'you', 'now', 'that', 'there', 'is', 'more', 'to', 'these', 'moons', 'than', 'meets', 'the', 'eye'], ['the', 'consolidation', 'they', 'hide', 'their', 'technology'], ['their', 'people', 'they', 'wont', 'tell', 'you', 'what', 'they', 'found', 'but', 'i', 'will'], ['one', 'moon', 'stands', 'apart', 'from', 'the', 'rest'], ['and', 'in', 'the', 'darkness', 'of', 'that', 'moon', 'an', 'ocean', 'of', 'blood']]


##1C. Stop-Word Removal
Description: Stop world removing, also known as filtering, is a phase of the text processing methods where we pick important words from the resulting tokens to represent the document. For my implementation, I will be using the stoplist method, this method removes undescriptive/unimportant words using a bag of words. The resulting tokens will be stored in the variable "filtered_tokens"

In [22]:
def remove_stop_words(tokens):
  # Example list of stop words, may add more later
  stop_words = [
    "a", "an", "the", "and", "or", "but", "if", "is", "are", "was", "were",
    "in", "on", "at", "of", "for", "to", "with", "as", "by", "from", "that",
    "this", "these", "those", "it", "its", "be", "been", "being", "have",
    "has", "had", "do", "does", "did", "will", "would", "shall", "should",
    "can", "could", "may", "might", "must", "about", "into", "over", "under",
    "above", "below", "between", "among", "so", "such", "no", "not", "too",
    "very", "just", "than", "then", "once", "here", "there", "when", "where",
    "why", "how", "all", "any", "both", "each", "few", "more", "most", "other",
    "some", "what", "which", "who", "whom", "whose", "also", "after", "before",
    "again", "further", "because", "while", "during", "until", "above", "below"
  ]

  filtered_tokens = []
  for token in tokens:
    #Only append if it's not in the stop word list
    if token not in stop_words:
      filtered_tokens.append(token)
  return filtered_tokens

filtered_documents = []
for tokens in tokenized_document:
  filtered_tokens = remove_stop_words(tokens)
  filtered_documents.append(filtered_tokens)
print(filtered_documents)

[['stars', 'gone'], ['planets', 'disappeared'], ['they', 'barely', 'enough', 'us', 'left', 'give', 'end', 'name'], ['quiet', 'rapture'], ['supplies', 'dwindle'], ['infrastructure', 'crumbles'], ['rebuild'], ['many', 'feed'], ['humanity', 'decays'], ['dont', 'despair', 'my', 'sons'], ['i', 'tell', 'you', 'now', 'moons', 'meets', 'eye'], ['consolidation', 'they', 'hide', 'their', 'technology'], ['their', 'people', 'they', 'wont', 'tell', 'you', 'they', 'found', 'i'], ['one', 'moon', 'stands', 'apart', 'rest'], ['darkness', 'moon', 'ocean', 'blood']]


## 1D. Preprocessing Function
Description: We can make a function preprocess() that wraps all of the previous methods to process each document in one call with ease. We can store the term indexes in the variable "term_indexes".

In [32]:
def preprocess(documents):
  processed_documents = []
  for document in documents:
    print("Original Document: ", document)

    # Lexical Analysis
    tokens = lexical_analysis(document)
    print("Lexical Analysis Output:", tokens)

    # Stop-word Removal
    filtered_tokens = remove_stop_words(tokens)
    print("Stop Word Removal Output:", filtered_tokens)
    processed_documents.append(filtered_tokens)

  return processed_documents

term_indexes = preprocess(documents)

Original Document:  The stars are gone.
Lexical Analysis Output: ['the', 'stars', 'are', 'gone']
Stop Word Removal Output: ['stars', 'gone']
Original Document:  The planets have disappeared.
Lexical Analysis Output: ['the', 'planets', 'have', 'disappeared']
Stop Word Removal Output: ['planets', 'disappeared']
Original Document:  They were barely enough of us left to give "the end" a name.
Lexical Analysis Output: ['they', 'were', 'barely', 'enough', 'of', 'us', 'left', 'to', 'give', 'the', 'end', 'a', 'name']
Stop Word Removal Output: ['they', 'barely', 'enough', 'us', 'left', 'give', 'end', 'name']
Original Document:  The Quiet Rapture.
Lexical Analysis Output: ['the', 'quiet', 'rapture']
Stop Word Removal Output: ['quiet', 'rapture']
Original Document:  Supplies dwindle.
Lexical Analysis Output: ['supplies', 'dwindle']
Stop Word Removal Output: ['supplies', 'dwindle']
Original Document:  Infrastructure crumbles.
Lexical Analysis Output: ['infrastructure', 'crumbles']
Stop Word Remova